# Milestone 3: Bank calls end-to-end
*AI from scratch, milestone 3* · combine notebook for Stage 3

Deliver an end-to-end production machine learning pipeline on direct marketing telemarketing data:
1. Inspect data and detect target leakage (`duration`).
2. Set up an honest, time-aware train/validation split.
3. Train a regularized logistic regression baseline with scikit-learn.
4. Train a modern gradient-boosted tree model (`HistGradientBoostingClassifier`).
5. Determine the profit-maximizing probability decision threshold from business costs.
6. Publish an actionable Model Card.

Replace each `...` under `# YOUR CODE HERE`, run the cell, then the check under it; `hint(n)` and `answer(n)`
are there when you're stuck.

In [ ]:
# Helpers: run this cell first. check() marks an answer ✓ or ✗; stuck? hint(n) gives the next hint for
# exercise n, answer(n) shows a solution.
import base64 as _b64
_HINTS = {1: ['y = (df["y"] == "yes").astype(int); X = df.drop(columns=["y", "duration"])'], 2: ['split = int(len(X) * 0.8); X_tr, X_te = X.iloc[:split], X.iloc[split:]; y_tr, y_te = y.iloc[:split], y.iloc[split:]'], 3: ["pre = ColumnTransformer([('num', StandardScaler(), num_cols), ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)])", "pipe_lr = Pipeline([('pre', pre), ('model', LogisticRegression(max_iter=1000))]).fit(X_tr, y_tr)", 'auc_lr = float(roc_auc_score(y_te, pipe_lr.predict_proba(X_te)[:, 1]))'], 4: ["pipe_gb = Pipeline([('pre', pre), ('model', HistGradientBoostingClassifier(random_state=42))]).fit(X_tr, y_tr)", 'auc_gb = float(roc_auc_score(y_te, pipe_gb.predict_proba(X_te)[:, 1]))'], 5: ['For thresh in np.arange(0.02, 0.42, 0.02): preds = (probs_gb >= thresh).astype(int); tp = int(((preds == 1) & (y_te == 1)).sum()); dialed = int((preds == 1).sum()); profit = 50 * tp - 2 * dialed; if profit > best_profit: ...'], 6: ['model_card = {"target": "term_deposit_subscription", "leak_prevention": "duration_dropped", "best_model": ..., "chosen_threshold": best_thresh, "validation_auc": round(max(auc_gb, auc_lr), 3)}']}
_ANSWERS = {1: 'eSA9IChkZlsieSJdID09ICJ5ZXMiKS5hc3R5cGUoaW50KQpYID0gZGYuZHJvcChjb2x1bW5zPVsieSIsICJkdXJhdGlvbiJdKQ==', 2: 'c3BsaXQgPSBpbnQobGVuKFgpICogMC44KQpYX3RyLCBYX3RlID0gWC5pbG9jWzpzcGxpdF0sIFguaWxvY1tzcGxpdDpdCnlfdHIsIHlfdGUgPSB5Lmlsb2NbOnNwbGl0XSwgeS5pbG9jW3NwbGl0Ol0=', 3: 'cHJlID0gQ29sdW1uVHJhbnNmb3JtZXIoWwogICAgKCdudW0nLCBTdGFuZGFyZFNjYWxlcigpLCBudW1fY29scyksCiAgICAoJ2NhdCcsIE9uZUhvdEVuY29kZXIoaGFuZGxlX3Vua25vd249J2lnbm9yZScsIHNwYXJzZV9vdXRwdXQ9RmFsc2UpLCBjYXRfY29scykKXSkKcGlwZV9sciA9IFBpcGVsaW5lKFsKICAgICgncHJlJywgcHJlKSwKICAgICgnbW9kZWwnLCBMb2dpc3RpY1JlZ3Jlc3Npb24obWF4X2l0ZXI9MTAwMCkpCl0pCnBpcGVfbHIuZml0KFhfdHIsIHlfdHIpCnByb2JzX2xyID0gcGlwZV9sci5wcmVkaWN0X3Byb2JhKFhfdGUpWzosIDFdCmF1Y19sciA9IGZsb2F0KHJvY19hdWNfc2NvcmUoeV90ZSwgcHJvYnNfbHIpKQ==', 4: 'cGlwZV9nYiA9IFBpcGVsaW5lKFsKICAgICgncHJlJywgcHJlKSwKICAgICgnbW9kZWwnLCBIaXN0R3JhZGllbnRCb29zdGluZ0NsYXNzaWZpZXIocmFuZG9tX3N0YXRlPTQyKSkKXSkKcGlwZV9nYi5maXQoWF90ciwgeV90cikKcHJvYnNfZ2IgPSBwaXBlX2diLnByZWRpY3RfcHJvYmEoWF90ZSlbOiwgMV0KYXVjX2diID0gZmxvYXQocm9jX2F1Y19zY29yZSh5X3RlLCBwcm9ic19nYikp', 5: 'Zm9yIHRocmVzaCBpbiBucC5hcmFuZ2UoMC4wMiwgMC40MiwgMC4wMik6CiAgICBwcmVkcyA9IChwcm9ic19nYiA+PSB0aHJlc2gpLmFzdHlwZShpbnQpCiAgICB0cCA9IGludCgoKHByZWRzID09IDEpICYgKHlfdGUgPT0gMSkpLnN1bSgpKQogICAgZGlhbGVkID0gaW50KChwcmVkcyA9PSAxKS5zdW0oKSkKICAgIHByb2ZpdCA9IDUwICogdHAgLSAyICogZGlhbGVkCiAgICBpZiBwcm9maXQgPiBiZXN0X3Byb2ZpdDoKICAgICAgICBiZXN0X3Byb2ZpdCA9IHByb2ZpdAogICAgICAgIGJlc3RfdGhyZXNoID0gcm91bmQoZmxvYXQodGhyZXNoKSwgMik=', 6: 'bW9kZWxfY2FyZCA9IHsKICAgICJ0YXJnZXQiOiAidGVybV9kZXBvc2l0X3N1YnNjcmlwdGlvbiIsCiAgICAibGVha19wcmV2ZW50aW9uIjogImR1cmF0aW9uX2Ryb3BwZWQiLAogICAgImJlc3RfbW9kZWwiOiAiSGlzdEdyYWRpZW50Qm9vc3RpbmciIGlmIGF1Y19nYiA+PSBhdWNfbHIgZWxzZSAiTG9naXN0aWNSZWdyZXNzaW9uIiwKICAgICJjaG9zZW5fdGhyZXNob2xkIjogYmVzdF90aHJlc2gsCiAgICAidmFsaWRhdGlvbl9hdWMiOiByb3VuZChtYXgoYXVjX2diLCBhdWNfbHIpLCAzKQp9'}
_given = {}

def check(n, test, msg=""):
    try:
        ok = bool(test())
    except NameError as e:
        return print("✗ %s: run the exercise cell above first." % e)
    except Exception as e:
        if "NoneType" in str(e) or "ellipsis" in str(e):
            return print("✗ Not answered yet: replace the ... in exercise %d and run that cell. Stuck? hint(%d)" % (n, n))
        ok, msg = False, msg + " (the check stopped with %s: %s)" % (type(e).__name__, e)
    print("✓ Right." if ok else "✗ Not yet. " + msg + "\n  Stuck? hint(%d) for a hint, answer(%d) for a solution." % (n, n))

def hint(n):
    hints, i = _HINTS.get(n, []), _given.get(n, 0)
    if not hints:
        return print("No hints for this one: answer(%d) shows a solution." % n)
    print("Hint %d of %d: %s" % (min(i, len(hints) - 1) + 1, len(hints), hints[min(i, len(hints) - 1)]))
    _given[n] = i + 1

def answer(n):
    print(_b64.b64decode(_ANSWERS[n]).decode())
    print("\n(Put this in place of the ..., run it, then close it and try writing it yourself.)")


In [ ]:
import io, urllib.request
import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix

# Load bank marketing dataset from repository
url = "https://raw.githubusercontent.com/ikruchynskyi/ihor.sh/main/ai/data/bank.csv"
raw_text = urllib.request.urlopen(url).read().decode()
sep = ";" if ";" in raw_text.splitlines()[0] else ","
df = pd.read_csv(io.StringIO(raw_text), sep=sep)
print(f"Loaded bank records: {len(df)} rows, {len(df.columns)} columns")
print("Target breakdown:", df["y"].value_counts().to_dict())

## 1. Detecting target leakage
The feature `duration` records the length of the phone call in seconds.
But our model must score customers *before* the sales agent dials the phone.
If `duration = 0`, the customer never picked up (`y = 0`). If `duration = 600`, the sales pitch has already succeeded.
Set `y = (df['y'] == 'yes').astype(int)` and create `X` by dropping both `"y"` and `"duration"`.

In [ ]:
# YOUR CODE HERE (stuck? hint(1), then answer(1))
...

In [ ]:
check(1, lambda: "duration" not in X.columns and "y" not in X.columns and len(X) == len(df) and y.isin([0, 1]).all(), "X excludes both the target and the leaking duration feature")

## 2. Time-aware train and validation split
Bank marketing campaigns run across sequential calendar months.
Split the data chronologically (first 80% for training, final 20% for validation).
Assign `X_tr`, `X_te`, `y_tr`, `y_te`.

In [ ]:
# YOUR CODE HERE (stuck? hint(2), then answer(2))
...

In [ ]:
check(2, lambda: len(X_tr) == int(len(df) * 0.8) and len(X_te) == len(df) - len(X_tr), "data is split chronologically into 80% train and 20% validation")

## 3. Logistic Regression baseline pipeline
Separate numeric and categorical column names:
Build a scikit-learn `Pipeline` with `ColumnTransformer` (standardizing numeric columns and one-hot encoding categoricals)
followed by `LogisticRegression(max_iter=1000)`.
Fit the pipeline on `(X_tr, y_tr)` and store validation ROC-AUC in `auc_lr`.

In [ ]:
feature_df = df.drop(columns=["y", "duration"], errors="ignore")
num_cols = list(feature_df.select_dtypes(include=[np.number]).columns)
cat_cols = list(feature_df.select_dtypes(exclude=[np.number]).columns)

# YOUR CODE HERE (stuck? hint(3), then answer(3))
...

if "auc_lr" in locals() and auc_lr is not None:
    print(f"Logistic Regression Validation ROC-AUC: {auc_lr:.3f}")

In [ ]:
check(3, lambda: auc_lr > 0.65, "Logistic Regression achieves solid baseline discriminatory power (AUC > 0.65)")

## 4. Boosted trees: HistGradientBoostingClassifier
Train a gradient-boosted tree model using `HistGradientBoostingClassifier(random_state=42)`.
Build a similar pipeline with `pre` and fit on `(X_tr, y_tr)`.
Store validation ROC-AUC in `auc_gb`.

In [ ]:
# YOUR CODE HERE (stuck? hint(4), then answer(4))
...

if "auc_gb" in locals() and auc_gb is not None:
    print(f"Gradient Boosted Trees Validation ROC-AUC: {auc_gb:.3f}")

In [ ]:
check(4, lambda: auc_gb > 0.65, "Gradient boosted trees achieve competitive validation AUC without target leakage")

## 5. Decision threshold from business costs
Accuracy defaults to a 0.5 probability threshold. In direct sales, costs and payoffs are asymmetric:
- Each dialed call costs \$2.00 (sales representative time).
- Each converted term deposit earns \$50.00 net profit.

Net profit for threshold $T$:
$$\text{Profit} = 50 \times \text{TP} - 2 \times (\text{TP} + \text{FP})$$

Test thresholds from 0.02 to 0.40 in steps of 0.02.
Store the threshold that maximizes total profit on the validation set in `best_thresh`.

In [ ]:
best_thresh = None
best_profit = -float("inf")

# YOUR CODE HERE (stuck? hint(5), then answer(5))
...

if "best_thresh" in locals() and best_thresh is not None:
    print(f"Optimal decision threshold: {best_thresh} (yields ${best_profit:,.2f} validation profit)")

In [ ]:
check(5, lambda: best_thresh is not None and 0.02 <= best_thresh <= 0.20, "Optimal cost-aware threshold is well below the naive 0.5 default, reflecting the 50:2 reward-to-cost ratio")

## 6. The Model Card
Summarize the system in a structured dictionary `model_card`:
- `"target"`: `"term_deposit_subscription"`
- `"leak_prevention"`: `"duration_dropped"`
- `"best_model"`: `"HistGradientBoosting" if auc_gb >= auc_lr else "LogisticRegression"`
- `"chosen_threshold"`: `best_thresh`
- `"validation_auc"`: `round(max(auc_gb, auc_lr), 3)`

In [ ]:
# YOUR CODE HERE (stuck? hint(6), then answer(6))
...

if "model_card" in locals() and model_card is not None:
    print("Model Card:", model_card)

In [ ]:
check(6, lambda: isinstance(model_card, dict) and model_card["leak_prevention"] == "duration_dropped" and "validation_auc" in model_card, "Model card documents target definition, leakage auditing, metric performance, and cost-calibrated decision threshold")